# 🤖 Estudo Prático de Agentes de IA com a API do Google Gemini

Este notebook faz parte dos meus estudos em **Inteligência Artificial e Engenharia de Prompt**, focado na criação de um **Sistema Multi-Agente** encadeado para automação de criação de conteúdo (posts de Instagram sobre tecnologia).

### 🎯 Objetivos de Aprendizado:
- Entender o conceito de **Agentes Especialistas** (Buscador, Planejador, Redator e Revisor).
- Utilizar a nova SDK oficial `google-genai` do Gemini.
- Aplicar **Google Search Grounding** para conectar o agente a buscas web em tempo real.
- Encadear as saídas de cada agente como entrada para o próximo na pipeline.

## 1. Instalação e Configuração das Dependências

In [ ]:
%pip install -q google-genai python-dotenv

In [ ]:
import os
from datetime import date
from google import genai
from google.genai import types

# Configuração do cliente Gemini SDK
# Defina sua variável de ambiente GOOGLE_API_KEY no arquivo .env ou no Colab Secrets
client = genai.Client()
MODEL_ID = "gemini-2.5-flash"

## 2. Definição dos Agentes Especialistas

O sistema é composto por 4 agentes que trabalham em sequência:

### Agente 1: Buscador de Notícias (com Google Search Grounding)

In [ ]:
def agente_buscador(topico: str, data_de_hoje: str) -> str:
    """Busca informações e novidades recentes no Google sobre o tema."""
    prompt = f"""
    Você é um pesquisador especialista em tendências de tecnologia.
    Pesquise no Google sobre o seguinte tópico: {topico}
    Considere a data atual como: {data_de_hoje}
    
    Resuma as notícias, lançamentos e fatos mais relevantes sobre este assunto.
    """
    
    # Ativa a ferramenta de busca do Google (Search Grounding)
    config = types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]
    )
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config=config
    )
    return response.text

### Agente 2: Planejador de Conteúdo

In [ ]:
def agente_planejador(topico: str, lancamentos_buscados: str) -> str:
    """Cria um roteiro estruturado com base nas notícias encontradas."""
    prompt = f"""
    Você é um estrategista de conteúdo para redes sociais.
    Tópico: {topico}
    Pesquisas encontradas:
    {lancamentos_buscados}
    
    Crie um plano estruturado para um post de Instagram contendo:
    1. Gancho (Hook): Frase marcante para prender a atenção.
    2. Pontos Chave: 2 a 3 tópicos centrais a serem explicados.
    3. Chamada para Ação (CTA): Pergunta engajadora ao final.
    """
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt
    )
    return response.text

### Agente 3: Redator Criativo

In [ ]:
def agente_redator(topico: str, plano_de_post: str) -> str:
    """Escreve a primeira versão do post com emojis e hashtags."""
    prompt = f"""
    Você é um Redator Criativo especialista em tecnologia.
    Tópico: {topico}
    Plano do Post:
    {plano_de_post}
    
    Escreva um rascunho de post para o Instagram com tom leve e amigável,
    utilizando emojis e finalizando com 2 a 4 hashtags relevantes.
    """
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt
    )
    return response.text

### Agente 4: Revisor de Qualidade

In [ ]:
def agente_revisor(topico: str, rascunho_gerado: str) -> str:
    """Faz a revisão final de tom de voz, clareza e gramática."""
    prompt = f"""
    Você é um Editor e Revisor de Conteúdo sênior para redes sociais.
    Público-alvo: Jovens e entusiastas de tecnologia (18 a 30 anos).
    
    Tópico: {topico}
    Rascunho:
    {rascunho_gerado}
    
    Revise a clareza, concisão e correção gramatical do texto.
    Retorne o texto final aprimorado e pronto para publicação.
    """
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt
    )
    return response.text

## 3. Execução da Pipeline Multi-Agente

In [ ]:
# Exemplo de execução da pipeline completa
topico_estudo = "Agentes de IA e Gemini 2.5"
data_atual = date.today().strftime("%d/%m/%Y")

print(f"🚀 Iniciando Sistema Multi-Agente para: '{topico_estudo}'\n")

# Step 1: Pesquisa de Notícias
print("[1/4] Buscando novidades...")
pesquisa = agente_buscador(topico_estudo, data_atual)

# Step 2: Planejamento
print("[2/4] Planejando post...")
plano = agente_planejador(topico_estudo, pesquisa)

# Step 3: Redação
print("[3/4] Redigindo rascunho...")
rascunho = agente_redator(topico_estudo, plano)

# Step 4: Revisão Final
print("[4/4] Revisando post...")
post_final = agente_revisor(topico_estudo, rascunho)

print("\n================ RESULTADO FINAL ================\n")
print(post_final)